# Herhalingslabo Classificatie

Hoe weet je wanneer je welk algoritme best gebruikt ? Gebruik volgende handleiding:

**KNearestNeighbours**
- Geschikt voor classificatie en regressie
- Voor kleine/middelgrote datasets
- heel simpel, kan snel worden getrained
- werkt niet goed voor datasets met heel veel features

**Support Vector Machines**
- Vooral een standaard voor binaire classificatie, of wanneer je een duidelijke marge wilt op classificatie (=het hoofddoel van het algoritme)
- Voor kleine/middelgrote datasets
- Is kostelijker om te trainen dan KNN, maar classificeert dan wel sneller
- Kernel keuze en hyperparameter tuning zijn essentieel

**Decision Trees**
- Wanneer je een 'white box', 'explainable' model wilt
- kan zowel numerieke als categorische features makkelijk aan
- kan heel goed om met missende datapunten
- het risico op overfitting is groot. Bij twijfel, gebruik een random forest

**Random forest**
- Zelfde voordelen als decision tree, minder risico op overfitting
- Geschikt voor middelgrote/grote datasets
- Geschikt voor data met (heel) veel features
- Hebben uiteraard meer trainingstijd nodig dan een enkele decision tree

**Adaboost**
- Als data niet goed is gebalanceerd (niet evenveel items van elke klasse)
- Vermijd een onderfit. Dit model is beter dan 1 enkele onderliggende decision tree

**XGBoost**
- XGBoost is bijna *altijd* een goede keuze
- Middelgrote / grote datasets
- Datasets met (heel) veel features
- De implementatie van XGBoost staat, ondanks dat het een boosting algoritme is, toch deels parallellisatie toe.

In de praktijk zijn er vaak meerdere goede keuzes. Het is belangrijk te weten waarom een bepaald model wel/niet een goede keuze zou zijn, en vervolgens de meest veelbelovende modellen te proberen en te kijken wat het beste werkt.

### Wat is een kleine / middelgrote / grote dataset ?

Uiteraard is dat een beetje afhankelijk van hoelang je bereid bent te wachten. In deze cursus gebruiken we de gangbare regel:
- Klein: <10k
- Middelgroot: 10k - 100k
- Groot: >100k

Dit is ongetwijfeld over een paar jaar volledig voorbijgestreefd. Besef dat je in real life dit altijd moet afwegen tegenover de beschikbare compute power, op een Raspberry Pi 3 zal een middelgrote dataset al een stevige trainingstijd hebben, terwijl dat kinderspel is op een cloud compute server.

## Oefeningen

Behandel de volgende 3 datasets - je kan ze downloaden vanop Digitap. Lees in, bekijk, en beoordeel welke algoritmes het meeste zin hebben. Pas de ML algoritmes toe en rapporteer de uiteindelijke score van je model.
- Diabetes - i de 'outcome' kolom wordt aangegeven of mensen diabetes hebben
- Customer Churn: in commerciële setting wordt vaak gesproken van 'churn', oftewel het aangeven in een soort diagram waar klanten in een process afhaken. Deze dataset bevat een 'churn' kolom die dat codeert.
- Creditcard bevat data die reeds door een ander algoritme is gegaan, waardoor de kolommen geen herkenbare naam meer hebben. De laatste kolom wijst op fraude (1) of een valide transactie (0).

In [ ]:
import numpy as np
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.ensemble import BaggingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

diabetes = pd.read_csv("./diabetes.csv")

diabetes_X = diabetes.drop(columns=["Outcome"])
diabetes_y = diabetes["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(diabetes_X, diabetes_y, random_state=42, test_size=0.2)

# diabetes.describe() laat toe om de hoeveelheid records/features van de dataset te zien
# print(diabetes.describe())

# ===================== DIABETES =====================
# Diabetes is een zeer kleine dataset met maar 769 records en 9 features. Hierom wordt dit geklassificeerd als een kleine dataset.
# Om te kijken of deze data lineair seperabel is gaan we werken met een support vector machine.
# TODO voor kleine datasets zoals deze is het beter om kleinere algoritmes te gebruiken zoals: KNN en SVC!
# we gebruiken voor (waar nodig, zoals SVC) een standardscaler omdat sommige algoritmes, waaronder SVC, gevoelig zijn aan de scale van de data
# ============== SVC
scaler = StandardScaler()

# diabetes_X = StandardScaler().fit_transform(diabetes_X) !! Niet meer nodig aangezien we dit al in de pipeline zetten.
X_train, X_test, y_train, y_test = train_test_split(diabetes_X, diabetes_y, random_state=42, test_size=0.2)

# clf_diabetes = SVC(kernel="linear")
# clf_diabetes = clf_diabetes.fit(X_train, y_train)
# we maken hier een pipeline van om de code wat orderlijker te maken
pipe = make_pipeline(StandardScaler(), SVC(random_state=10, kernel="linear")).fit(X_train, y_train)

y_pred_diabetes = pipe.predict(X_test)
y_accuracy_diabetes = accuracy_score(y_pred_diabetes, y_test)
print(f"y_accuracy_diabetes SUPPORT VECTOR MACHINE: {y_accuracy_diabetes}")

print()

print(confusion_matrix(y_test, y_pred_diabetes))
print(classification_report(y_test, y_pred_diabetes))

# ============== Decision tree 
# TODO niet goed! eigenlijk een te kleine dataset voor een decision tree!
clf_diabetes = DecisionTreeClassifier(random_state=42)
clf_diabetes = clf_diabetes.fit(X_train, y_train)

y_pred_diabetes = clf_diabetes.predict(X_test)
y_accuracy_diabetes = accuracy_score(y_pred_diabetes, y_test)
print(f"y_accuracy_diabetes DECISION TREE: {y_accuracy_diabetes}")

# decision model is natuurlijk prone aan overfitten, we gaan hier dus een XGB op testen om te kijken of het probleem bij de data of het model ligt
# ============== XGBoost 
xgb_classifier = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
xgb_classifier.fit(X_train, y_train)

y_pred_xgb_diabetes = xgb_classifier.predict(X_test)
y_accuracy_xgb_diabetes = accuracy_score(y_pred_xgb_diabetes, y_test)

print(f"y_accuracy_diabetes XGBOOST: {y_accuracy_xgb_diabetes}")

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier


# ===================== Telco Customer Churn =====================
# door telco.describe() zien we het aantal records =et 7000+ en ongeveer 20 features dus dit is een middelgrote dataset
# hier is het mogelijks goed om een te kijken naar algoritmes zoals XGBoost of een random forest, al is het nog altijd wel mogelijk om SVC/KNN te gebruiken.
# ============== preprocessing
# Bij het bekijken van de dataset is het al snel te zien dat heel veel van de data niet encoded is dus dit moeten we oplossen.
# print(telco.head())
# print(telco.columns)

telco = pd.read_csv("./Telco-Customer-Churn.csv")

telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
print()
# zijn er enige null values
print(telco.isna().sum())
print()
telco_X = telco.drop(columns=["Churn", "customerID"])
telco_y = LabelEncoder().fit_transform(telco["Churn"])

# Preprocessing voor numerieke features
cat_cols = telco_X.select_dtypes(include="object").columns
num_cols = telco_X.select_dtypes(exclude="object").columns

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
])


# encoder = LabelEncoder()

# ============== SVC
# Redenering: KNN werkt niet goed op datasets met veel features ,

model = make_pipeline(preprocess, SVC(random_state=42, kernel="linear"))

X_train, X_test, y_train, y_test = train_test_split(telco_X, telco_y, random_state=42, test_size=0.2)

model.fit(X_train, y_train)
# pipe = make_pipeline(StandardScaler(), SVC(kernel="linear", random_state=42)).fit(X_train, y_train)
y_pred_telco_svc = model.predict(X_test)
y_accuracy_telco_sbc = accuracy_score(y_pred_telco_svc, y_test)

print(f"y_accuracy_telco_sbc {y_accuracy_telco_sbc}")

# ============== XGBoost

model = make_pipeline(preprocess, XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
model.fit(X_train, y_train)
y_pred_telco_svc = model.predict(X_test)
y_accuracy_telco_XGB = accuracy_score(y_pred_telco_svc, y_test)

print(f"y_accuracy_telco_XGB {y_accuracy_telco_sbc}")

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ===================== creditcard =====================
# we zien hier een tijd-kolom in!! Dit is dus een textbook case om ARIMA te gebruiken.
credit = pd.read_csv("./creditcard.csv")
X = credit.drop(columns=["Class"])
y = credit["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ZWAAR ONGEBALANCEERD!!!!!!! oplossing: ADABOOST
credit2 = credit["Class"].value_counts()
print(credit2)

ada_classifier = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_classifier.fit(X_train, y_train)

y_pred_ada = ada_classifier.predict(X_test)

print(confusion_matrix(y_test, y_pred_ada))
print(accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))
